# NMR Graph Matching - Example Usage

This notebook demonstrates how to use the NMR graph matching pipeline for methyl assignment.

In [ ]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import matplotlib.pyplot as plt

from nmr_graph_matching import (
    PDBParser,
    HMQCParser,
    NOESYParser,
    MethylNetworkBuilder,
    PeakNetworkBuilder,
    DGMCModel,
    OutputFormatter
)

## 1. Parse Input Files

Load PDB structure and NMR peak lists.

In [ ]:
# File paths (update these to your data)
pdb_file = "../data/raw/example.pdb"
hmqc_file = "../data/raw/example_hmqc.txt"
noesy_file = "../data/raw/example_noesy.txt"

# Parse PDB to extract methyl groups
print("Parsing PDB file...")
pdb_parser = PDBParser()
methyl_groups = pdb_parser.parse(pdb_file)

print(f"Found {len(methyl_groups)} methyl groups")
print("\nSummary by residue type:")
summary = pdb_parser.get_summary()
for res_type, count in summary.items():
    print(f"  {res_type}: {count}")

In [ ]:
# Parse HMQC peak list
print("Parsing HMQC peak list...")
hmqc_parser = HMQCParser()
hmqc_peaks = hmqc_parser.parse(hmqc_file)

print(f"Found {len(hmqc_peaks)} HMQC peaks")

# Display first few peaks
df = hmqc_parser.get_peak_dataframe()
print("\nFirst 5 peaks:")
print(df.head())

In [ ]:
# Parse NOESY peak list and match with HMQC
print("Parsing NOESY peak list...")
noesy_parser = NOESYParser(tolerance=0.05)
noesy_peaks = noesy_parser.parse(noesy_file)

print(f"Found {len(noesy_peaks)} NOESY peaks")

# Match NOESY peaks with HMQC to identify cross-peaks
print("\nMatching NOESY with HMQC...")
crosspeaks = noesy_parser.match_with_hmqc(hmqc_peaks)

print(f"Identified {len(crosspeaks)} NOE cross-peaks")

## 2. Build Graphs

Construct the methyl network (Graph B) and peak network (Graph A).

In [ ]:
# Build methyl network
print("Building methyl network...")
methyl_builder = MethylNetworkBuilder(
    distance_cutoff=10.0,  # Angstroms
    min_distance=0.0
)
methyl_graph = methyl_builder.build(methyl_groups)

print(f"Methyl network: {methyl_graph.num_nodes} nodes, {methyl_graph.edge_index.size(1)} edges")
print(f"Node feature dimension: {methyl_graph.x.size(1)}")
print(f"Edge feature dimension: {methyl_graph.edge_attr.size(1)}")

In [ ]:
# Build peak network
print("Building peak network...")
peak_builder = PeakNetworkBuilder(
    shift_normalization="standard",
    confidence_threshold=0.0
)
peak_graph = peak_builder.build(hmqc_peaks, crosspeaks)

print(f"Peak network: {peak_graph.num_nodes} nodes, {peak_graph.edge_index.size(1)} edges")
print(f"Node feature dimension: {peak_graph.x.size(1)}")
print(f"Edge feature dimension: {peak_graph.edge_attr.size(1)}")

## 3. Visualize Graphs (Optional)

Visualize the constructed graphs to understand the data.

In [ ]:
# Visualize methyl network in 3D
methyl_builder.visualize_network(methyl_graph)

In [ ]:
# Visualize peak network
peak_builder.visualize_network(peak_graph, layout='spring')

## 4. Load Model and Predict

Load a trained model and predict methyl assignments.

In [ ]:
# Create model
model = DGMCModel(
    peak_feature_dim=4,
    methyl_feature_dim=11,
    embedding_dim=64,
    hidden_dim=128,
    num_encoder_layers=3,
    num_consensus_layers=2,
    gnn_type='gcn'
)

# Load checkpoint (if available)
checkpoint_path = "../checkpoints/best_model.pt"
try:
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded model from {checkpoint_path}")
except:
    print("No checkpoint found. Using untrained model (for demonstration only).")

model.eval()

In [ ]:
# Predict assignments
print("Running prediction...")
with torch.no_grad():
    matching_matrix, (peak_emb, methyl_emb) = model(
        peak_graph, 
        methyl_graph, 
        return_embeddings=True
    )

print(f"Matching matrix shape: {matching_matrix.shape}")
print(f"Peak embeddings shape: {peak_emb.shape}")
print(f"Methyl embeddings shape: {methyl_emb.shape}")

## 5. Format and Display Results

In [ ]:
# Format assignments
formatter = OutputFormatter(
    confidence_threshold=0.5,
    top_k_alternatives=3
)

# Get peak shifts for output
peak_shifts = np.column_stack([
    [p.shifts[0] for p in hmqc_peaks],  # H shifts
    [p.shifts[1] for p in hmqc_peaks]   # C shifts
])

# Get adjacency matrices for NOE completeness
peak_adj = peak_builder.get_adjacency_matrix(peak_graph)
methyl_adj = methyl_builder.get_adjacency_matrix(methyl_graph)

results = formatter.format_assignments(
    matching_matrix=matching_matrix,
    peak_ids=peak_graph.peak_ids,
    peak_shifts=peak_shifts,
    methyl_names=methyl_graph.methyl_names,
    peak_adjacency=peak_adj,
    methyl_adjacency=methyl_adj
)

print(f"Generated {len(results)} assignments")

In [ ]:
# Display as text
text_output = formatter.to_text(results, include_alternatives=True, only_confident=False)
print(text_output)

In [ ]:
# Display as DataFrame
df = formatter.to_dataframe(results)
df

## 6. Analyze Matching Matrix

Visualize the predicted matching scores.

In [ ]:
# Visualize matching matrix
plt.figure(figsize=(12, 10))
plt.imshow(matching_matrix.detach().cpu().numpy(), aspect='auto', cmap='viridis')
plt.colorbar(label='Matching score')
plt.xlabel('Methyl index')
plt.ylabel('Peak index')
plt.title('Predicted Matching Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze confidence distribution
confidences = [r.confidence for r in results]

plt.figure(figsize=(10, 6))
plt.hist(confidences, bins=20, edgecolor='black', alpha=0.7)
plt.axvline(0.5, color='red', linestyle='--', label='Threshold')
plt.xlabel('Confidence')
plt.ylabel('Number of assignments')
plt.title('Assignment Confidence Distribution')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

high_conf = sum(1 for c in confidences if c >= 0.5)
print(f"High-confidence assignments (≥0.5): {high_conf}/{len(confidences)} ({100*high_conf/len(confidences):.1f}%)")

## 7. Save Results

In [ ]:
# Save to CSV
formatter.to_csv(results, "../results/assignments.csv", include_alternatives=True)

# Save text report
with open("../results/assignments.txt", 'w') as f:
    f.write(text_output)

# Generate PyMOL script
formatter.generate_pymol_script(
    results=results,
    pdb_file=pdb_file,
    output_file="../results/assignments.pml"
)

print("Results saved to ../results/")